In [1]:
# Lesson 4: Feature Engineering
# Project: Placement Analytics & Prediction System

import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv('../data/processed/placement_clean.csv')

print("Dataset loaded!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

Dataset loaded!
Shape: (215, 14)

Columns: ['gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s', 'degree_p', 'degree_t', 'workex', 'etest_p', 'specialisation', 'mba_p', 'status', 'salary']


In [ ]:
#Step 1: Drop salary and separate target

#Drop salary column
df = df.drop(columns=['salary'])

#seperating features and target
X = df.drop(columns=['status'])     #everything except status
Y = df['status']                    #only status column

print(f'Feature shape: {X.shape}')
print(f"Target shape: {Y.shape}")
print(f"\nTarget Values: {Y.unique()}")

Feature shape: (215, 12)
Target shape: (215,)

Target Values: <ArrowStringArray>
['Placed', 'Not Placed']
Length: 2, dtype: str


In [ ]:
#step 2: Identify categorical columns

categorical_cols = X.select_dtypes(include='str').columns.tolist() #finds all text/string columns and converts result to a simple Python list
numerical_cols = X.select_dtypes(include='number').columns.tolist()

print(f"Categorical coolumns: {categorical_cols}")      #it needs encoding
print(f"Numerical columns: {numerical_cols}")           #scaling

Categorical coolumns: ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation']
Numerical columns: ['ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p']


In [6]:
#Step 3: Label Encoding

#Define mapping for each binary column
label_mapping = {
    'gender'        : {'M' : 1, 'F' : 0},
    'workex'        : {'Yes' : 1, 'No' : 0},
    'ssc_b'         : {'Central' : 1, 'Others' : 0},
    'hsc_b'         : {'Central' : 1, 'Others' : 0},
    'specialisation': {'Mkt&Fin': 1, 'Mkt&HR': 0},
}

#Apply maaping to each col
for col, mapping in label_mapping.items():  #loops through each column one by one
    X[col] = X[col].map(mapping)            #replaces text values with numbers based on our mapping

print(X[['gender', 'workex', 'ssc_b', 'hsc_b', 'specialisation']].head())

   gender  workex  ssc_b  hsc_b  specialisation
0       1       0      0      0               0
1       1       1      1      0               1
2       1       0      1      1               1
3       1       0      1      1               0
4       1       0      1      1               1


In [8]:
#Step 4: One hot encoding

X = pd.get_dummies(X, columns=['hsc_s', 'degree_t'], drop_first=True)
#pd.get_dummies() → Pandas built-in function for One Hot Encoding
#drop_first=True → drops the first category column to avoid redundancy

print(f"New shape: {X.shape}")
print(f"\nNew Columns: {list(X.columns)}")

New shape: (215, 14)

New Columns: ['gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'degree_p', 'workex', 'etest_p', 'specialisation', 'mba_p', 'hsc_s_Commerce', 'hsc_s_Science', 'degree_t_Others', 'degree_t_Sci&Tech']


In [9]:
#Step 5: Encode Target variable

Y = Y.map({'Placed' : 1, 'Not Placed' : 0})

print(f"Unique Values: {Y.unique()}")
print(f"\nValue counts: {Y.value_counts()}")

Unique Values: [1 0]

Value counts: status
1    148
0     67
Name: count, dtype: int64


In [10]:
#Step 6: Scalling Numerical Column
from sklearn.preprocessing import MinMaxScaler  #imports the scaller

scaller = MinMaxScaler()        #creates a scaler object

X[numerical_cols] = scaller.fit_transform(X[numerical_cols])
#fit_transform() → learns the min/max values AND scales at the same time

print(X[numerical_cols].head())

      ssc_p     hsc_p  degree_p   etest_p     mba_p
0  0.538240  0.889621  0.195122  0.104167  0.284483
1  0.792414  0.680890  0.670244  0.760417  0.564843
2  0.497011  0.510708  0.341463  0.520833  0.247001
3  0.311482  0.247117  0.048780  0.333333  0.308096
4  0.925788  0.602965  0.568293  0.975000  0.160795


In [11]:
#Step 7: Final Check

print("Final Feature Matrix (X):")
print(f"Shape: {X.shape}")
print(f"\nColumns: {list(X.columns)}")
print(f"\nFirst 5 rows:")
print(X.head())

print(f"\nTarget Variable (Y):")
print(Y.head())


Final Feature Matrix (X):
Shape: (215, 14)

Columns: ['gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'degree_p', 'workex', 'etest_p', 'specialisation', 'mba_p', 'hsc_s_Commerce', 'hsc_s_Science', 'degree_t_Others', 'degree_t_Sci&Tech']

First 5 rows:
   gender     ssc_p  ssc_b     hsc_p  hsc_b  degree_p  workex   etest_p  \
0       1  0.538240      0  0.889621      0  0.195122       0  0.104167   
1       1  0.792414      1  0.680890      0  0.670244       1  0.760417   
2       1  0.497011      1  0.510708      1  0.341463       0  0.520833   
3       1  0.311482      1  0.247117      1  0.048780       0  0.333333   
4       1  0.925788      1  0.602965      1  0.568293       0  0.975000   

   specialisation     mba_p  hsc_s_Commerce  hsc_s_Science  degree_t_Others  \
0               0  0.284483            True          False            False   
1               1  0.564843           False           True            False   
2               1  0.247001           False          False    

In [ ]:
#Fix: Convert boolean columns to integers

bool_cols = X.select_dtypes(include='bool').columns.tolist()    # finds all boolean columns
X[bool_cols] = X[bool_cols].astype(int)                         # converts True → 1 and False → 0

print(X[['hsc_s_Commerce', 'hsc_s_Science', 'degree_t_Others', 'degree_t_Sci&Tech']].head())

   hsc_s_Commerce  hsc_s_Science  degree_t_Others  degree_t_Sci&Tech
0               1              0                0                  1
1               0              1                0                  1
2               0              0                0                  0
3               0              1                0                  1
4               1              0                0                  0


In [ ]:
#Step 8: Save Prepared Data

X.to_csv('../data/processed/X_prepared.csv', index=False)
Y.to_csv('../data/processed/Y_prepared.csv', index=False)

print("✅ Prepared data saved!")
print(f"X saved: data/processed/X_prepared.csv")
print(f"Y saved: data/processed/y_prepared.csv")

✅ Prepared data saved!
X saved: data/processed/X_prepared.csv
Y saved: data/processed/y_prepared.csv
